In [1]:
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python311.zip')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/lib-dynload')
sys.path.append('/home/amunif/.local/lib/python3.11/site-packages')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/site-packages')

In [2]:
from sklearn.datasets import make_classification
import numpy as np

import xgboost as xgb

# Using Numpy matrix

In [3]:
# Make a synthetic ranking dataset for demonstration
seed = 1994
X, y = make_classification(random_state=seed)
rng = np.random.default_rng(seed)
n_query_groups = 1 # Single query group
qid = rng.integers(0, n_query_groups, size=X.shape[0])

In [4]:
X

array([[ 1.37274475,  0.55556022, -0.39472313, ..., -1.52678928,
        -0.95222834, -0.58304118],
       [ 0.29797914,  0.60418421,  0.24888156, ..., -0.68180427,
        -0.81937699, -1.8177727 ],
       [-0.20681544,  0.18770098, -0.31246672, ..., -1.71367001,
        -1.77090831, -0.37479097],
       ...,
       [-0.57437761, -1.66850427,  1.18792669, ..., -0.31096033,
         0.80405855,  0.25831555],
       [ 0.86229379,  0.61644239,  1.7479823 , ..., -0.23655313,
        -1.26747288, -0.20751249],
       [-0.34736961, -0.81172299, -0.21660061, ...,  0.05799518,
        -0.45621093,  0.36914458]])

In [5]:
# Sort the inputs based on query index
sorted_idx = np.argsort(qid)
X = X[sorted_idx, :]
y = y[sorted_idx]
qid = qid[sorted_idx]

In [6]:
ranker = xgb.XGBRanker(
            tree_method="hist", 
            lambdarank_num_pair_per_sample=8, 
            objective="rank:pairwise", 
            lambdarank_pair_method="topk"
        )

ranker.fit(X, y, qid=qid)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=8, lambdarank_pair_method='topk',
          learning_rate=None, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, ...)

In [7]:
y

array([0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0,
       0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0])

In [8]:
y_pred = ranker.predict(X)

In [9]:
y_pred[:25]

array([-3.111158  , -3.111158  ,  2.1305983 ,  2.159494  , -0.24128146,
       -2.8260896 , -3.111158  , -2.7835517 , -3.111158  , -3.111158  ,
        0.48976067,  1.8862458 , -2.1204765 ,  1.60456   , -3.111158  ,
       -3.111158  ,  1.6566339 , -3.111158  , -2.7835517 , -2.8260896 ,
       -3.111158  ,  2.9567578 , -3.111158  ,  1.5222495 , -2.4984834 ],
      dtype=float32)

In [10]:
y[:25]

array([0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 1, 1])

In [11]:
qid[:25]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0])

# Using Pandas Dataframe

In [12]:
import pandas as pd

# `X`, `qid`, and `y` are from the previous snippet, they are all sorted by the `sorted_idx`.
df = pd.DataFrame(X, columns=[str(i) for i in range(X.shape[1])])
df["qid"] = qid

In [13]:
df

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,qid
0,1.372745,0.555560,-0.394723,-0.716318,1.193084,-0.746659,-1.325759,-1.036337,-0.121115,-0.942284,...,-0.868152,-2.116384,-0.389823,-1.007291,-1.291852,0.184040,-1.526789,-0.952228,-0.583041,0
1,0.815276,-0.668167,1.709072,0.949178,1.679298,0.413833,-1.045914,-0.770726,-0.303531,1.084170,...,-0.862140,0.068191,-0.182288,-0.202916,-1.298151,0.763849,0.338292,-0.469184,-1.510361,0
2,-0.410220,-0.087918,-0.269315,-0.991832,-1.606111,-0.056275,1.737854,1.890486,-0.101579,0.123377,...,0.930753,-0.198029,-0.957908,1.107039,0.904915,-1.384724,0.456717,-2.231578,-1.713816,0
3,-0.438941,-0.518367,-0.414652,-1.031209,-0.432312,1.252436,0.024844,0.539195,0.270635,0.303833,...,-0.073392,-0.855536,-0.975249,1.240300,0.063219,-1.498230,0.180950,1.129922,-1.040233,0
4,-0.206068,0.527394,0.785063,2.222237,0.780396,0.505323,-1.813242,-0.213693,-0.188778,-1.282348,...,0.182024,-0.604494,0.921076,1.735527,-0.151414,0.339606,0.310410,0.326105,0.706655,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.776900,0.749882,0.000656,-0.069825,0.778200,-0.103206,-1.018245,1.063971,0.276800,0.585842,...,-1.303239,1.476412,-0.434841,-1.036898,-0.021033,0.633131,-0.161629,0.601452,-0.135858,0
96,1.573806,-0.395907,-0.235761,-1.295802,-0.224563,-0.323659,-0.446189,-0.846448,2.498920,-0.465098,...,0.898972,-1.273912,0.019470,1.885331,2.059463,-2.096817,-0.100299,-0.527940,-0.115722,0
97,-0.073683,0.403909,-1.143694,0.689512,-1.743075,0.426403,-2.367238,-1.680840,1.428203,-0.205916,...,0.819539,0.016842,-0.581671,-2.603692,-0.422589,2.164634,-0.024611,3.361389,0.254388,0
98,-0.348024,1.122129,-0.674621,-0.967987,-0.388883,0.166560,2.517418,-0.248223,-1.036734,0.735238,...,0.834863,-1.873104,-0.517272,1.135228,-1.222280,-1.387349,1.676428,0.673324,1.221138,0


In [14]:
y

array([0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0,
       0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0])

In [15]:
ranker.fit(df, y)  # No need to pass qid as a separate argument

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=8, lambdarank_pair_method='topk',
          learning_rate=None, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, ...)

In [16]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
# Works with cv in scikit-learn, along with HPO utilities like GridSearchCV
kfold = StratifiedGroupKFold(shuffle=False)
print(cross_val_score(ranker, df, y, cv=kfold, groups=df.qid))

/group/pmc021/amunif/env/pytorch/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [11:12:08] WARNING: /workspace/src/common/error_msg.cc:52: Empty dataset at worker: 0
  warnings.warn(smsg, UserWarning)


[0.         0.         0.         0.32550785 0.        ]


In [17]:
scores = ranker.predict(X)

In [18]:
scores

array([-3.111158  , -3.111158  ,  2.1305983 ,  2.159494  , -0.24128146,
       -2.8260896 , -3.111158  , -2.7835517 , -3.111158  , -3.111158  ,
        0.48976067,  1.8862458 , -2.1204765 ,  1.60456   , -3.111158  ,
       -3.111158  ,  1.6566339 , -3.111158  , -2.7835517 , -2.8260896 ,
       -3.111158  ,  2.9567578 , -3.111158  ,  1.5222495 , -2.4984834 ,
       -1.7435392 ,  2.351162  ,  1.1608378 , -3.111158  ,  2.7630053 ,
        2.852698  ,  2.8125908 , -3.111158  , -2.8260896 , -1.7928706 ,
       -3.111158  , -3.111158  , -3.111158  , -2.7835517 ,  0.9883237 ,
       -2.8260896 , -2.7835517 ,  1.4769713 , -2.8210504 ,  0.4259906 ,
       -3.111158  , -1.9763489 , -3.111158  , -3.111158  , -2.8210504 ,
       -2.1204765 ,  2.4941015 , -2.7835517 , -1.981113  ,  1.7867619 ,
       -2.4934442 ,  0.7218866 , -2.8260896 , -2.8210504 ,  2.1615055 ,
        1.5753603 ,  1.7160127 , -2.8614388 , -2.8260896 , -3.111158  ,
       -1.7928706 ,  2.6417422 , -2.8260896 , -1.7435392 ,  1.83

In [19]:
df_full = df

In [20]:
df_full['y'] = y

In [21]:
df_full['score'] = scores

In [22]:
df_full

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,qid,y,score
0,1.372745,0.555560,-0.394723,-0.716318,1.193084,-0.746659,-1.325759,-1.036337,-0.121115,-0.942284,...,-0.389823,-1.007291,-1.291852,0.184040,-1.526789,-0.952228,-0.583041,0,0,-3.111158
1,0.815276,-0.668167,1.709072,0.949178,1.679298,0.413833,-1.045914,-0.770726,-0.303531,1.084170,...,-0.182288,-0.202916,-1.298151,0.763849,0.338292,-0.469184,-1.510361,0,0,-3.111158
2,-0.410220,-0.087918,-0.269315,-0.991832,-1.606111,-0.056275,1.737854,1.890486,-0.101579,0.123377,...,-0.957908,1.107039,0.904915,-1.384724,0.456717,-2.231578,-1.713816,0,1,2.130598
3,-0.438941,-0.518367,-0.414652,-1.031209,-0.432312,1.252436,0.024844,0.539195,0.270635,0.303833,...,-0.975249,1.240300,0.063219,-1.498230,0.180950,1.129922,-1.040233,0,1,2.159494
4,-0.206068,0.527394,0.785063,2.222237,0.780396,0.505323,-1.813242,-0.213693,-0.188778,-1.282348,...,0.921076,1.735527,-0.151414,0.339606,0.310410,0.326105,0.706655,0,0,-0.241281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.776900,0.749882,0.000656,-0.069825,0.778200,-0.103206,-1.018245,1.063971,0.276800,0.585842,...,-0.434841,-1.036898,-0.021033,0.633131,-0.161629,0.601452,-0.135858,0,1,-3.111158
96,1.573806,-0.395907,-0.235761,-1.295802,-0.224563,-0.323659,-0.446189,-0.846448,2.498920,-0.465098,...,0.019470,1.885331,2.059463,-2.096817,-0.100299,-0.527940,-0.115722,0,1,1.764576
97,-0.073683,0.403909,-1.143694,0.689512,-1.743075,0.426403,-2.367238,-1.680840,1.428203,-0.205916,...,-0.581671,-2.603692,-0.422589,2.164634,-0.024611,3.361389,0.254388,0,0,-2.826090
98,-0.348024,1.122129,-0.674621,-0.967987,-0.388883,0.166560,2.517418,-0.248223,-1.036734,0.735238,...,-0.517272,1.135228,-1.222280,-1.387349,1.676428,0.673324,1.221138,0,1,1.678591


In [23]:
df_full.sort_values(by=['qid', 'score'], ascending=[True, False])

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,qid,y,score
21,-1.792814,0.505197,-0.758400,-0.971713,0.448603,-0.501972,0.948744,1.001042,0.087772,0.089457,...,-0.961131,1.355219,1.864423,-1.533999,0.337681,-0.460447,-2.172854,0,1,2.956758
30,-1.842752,0.076834,-0.334669,-0.835040,1.243732,0.455474,-0.840762,-1.492886,-0.534771,-0.131939,...,-0.235151,1.345021,-0.342737,-1.436477,0.744842,1.800947,-0.268409,0,1,2.852698
31,-1.153610,0.971878,-0.129294,1.076606,0.724425,0.475259,0.086913,0.946241,0.108397,0.354858,...,-0.346356,1.519895,0.061803,-0.280516,0.005148,-0.478293,0.028333,0,1,2.812591
29,-1.106894,-1.096703,0.777550,1.297725,-0.756983,1.012394,-0.412962,-0.296868,-1.457080,-0.801765,...,1.046101,1.476697,0.196245,-0.105240,-0.655125,-0.226297,0.702951,0,1,2.763005
66,-2.104483,-1.120227,-0.174227,-1.157518,0.561643,0.369635,-1.946725,-1.741876,-0.515523,0.903832,...,-0.093665,1.370380,1.486202,-1.667430,-0.026358,-0.388866,0.031561,0,1,2.641742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,-0.206815,0.187701,-0.312467,0.106127,1.119420,0.921516,0.227782,-0.228953,-0.858692,-1.302184,...,0.877543,-0.774941,-1.615519,0.578401,-1.713670,-1.770908,-0.374791,0,1,-3.111158
79,1.888667,0.814382,0.402607,0.407186,0.150238,0.482765,-0.567565,-1.173693,0.254967,0.433210,...,-1.241469,-1.021642,-1.099080,0.940177,-0.759097,-1.060872,0.532731,0,0,-3.111158
81,0.207814,1.463490,-2.227515,0.474975,0.938369,-0.989747,-1.618113,-0.731918,0.143338,-1.179639,...,-1.085116,-0.375408,-0.300684,0.561717,-0.834832,-0.358877,-0.009439,0,0,-3.111158
88,-0.295513,0.516707,0.749158,2.578971,0.691794,-1.281326,-1.293071,-1.346305,-0.489739,1.366932,...,-1.646107,-0.446111,-0.333021,2.006463,0.997669,-0.079598,0.308927,0,0,-3.111158


In [24]:
df_full.sort_values(by=['score'], ascending=[False])

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,qid,y,score
21,-1.792814,0.505197,-0.758400,-0.971713,0.448603,-0.501972,0.948744,1.001042,0.087772,0.089457,...,-0.961131,1.355219,1.864423,-1.533999,0.337681,-0.460447,-2.172854,0,1,2.956758
30,-1.842752,0.076834,-0.334669,-0.835040,1.243732,0.455474,-0.840762,-1.492886,-0.534771,-0.131939,...,-0.235151,1.345021,-0.342737,-1.436477,0.744842,1.800947,-0.268409,0,1,2.852698
31,-1.153610,0.971878,-0.129294,1.076606,0.724425,0.475259,0.086913,0.946241,0.108397,0.354858,...,-0.346356,1.519895,0.061803,-0.280516,0.005148,-0.478293,0.028333,0,1,2.812591
29,-1.106894,-1.096703,0.777550,1.297725,-0.756983,1.012394,-0.412962,-0.296868,-1.457080,-0.801765,...,1.046101,1.476697,0.196245,-0.105240,-0.655125,-0.226297,0.702951,0,1,2.763005
66,-2.104483,-1.120227,-0.174227,-1.157518,0.561643,0.369635,-1.946725,-1.741876,-0.515523,0.903832,...,-0.093665,1.370380,1.486202,-1.667430,-0.026358,-0.388866,0.031561,0,1,2.641742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,-0.206815,0.187701,-0.312467,0.106127,1.119420,0.921516,0.227782,-0.228953,-0.858692,-1.302184,...,0.877543,-0.774941,-1.615519,0.578401,-1.713670,-1.770908,-0.374791,0,1,-3.111158
35,1.236995,-0.147629,-0.528348,1.893250,-0.793055,-2.114816,-0.659238,1.560416,-0.419167,0.142556,...,-0.755165,-0.103233,0.164145,1.325994,0.544517,1.368002,-1.539482,0,0,-3.111158
36,0.574299,-1.367357,0.042897,0.657376,1.174720,0.509925,-1.854971,-0.090408,-0.229770,1.927072,...,-2.471707,0.281096,-0.274928,0.252703,0.075428,-1.178797,-0.503624,0,0,-3.111158
37,-0.114905,-0.714108,-0.770644,2.200479,0.989420,-0.998984,0.655971,-0.570474,1.504251,1.132023,...,-0.452211,-0.166921,-0.601385,1.571931,0.751957,-1.565574,0.736031,0,0,-3.111158


In [25]:
df_full.sort_values(by=['score'], ascending=[False]).to_csv('df_full_ranking_single_group.csv', header=True, index=False)

## Sorting the scores

In [ ]:
# sorted_idx = np.argsort(scores)[::-1]
# # Sort the relevance scores from most relevant to least relevant
# scores = scores[sorted_idx]

In [ ]:
# scores